# Экперимент с Baseline++: baseline + enc7/dec5 + ffn1536

Надо скопировать этот ноутбук и выполнить общие ячейки, затем выбрать **одну из трёх веток**: с нуля, с весов или resume. У каждой ветки свои параметры, сетап и запуск. Для последовательного запуска нескольких новых экспериментов предусмотрен отдельный блок очереди в конце; он выполняется вместо одиночной ветки.

Обучение запускается на двух GPU T4 в Kaggle. Метрики отправляются в W&B, результаты сохраняются в `/kaggle/working/experiment_runs`.

## Общие пути

- `PROJECT_PATH`: репозиторий с `utilities/train_script.py` и конфигами.
- `DATA_PATH`: исходная папка `tokenized` с `dataset_dict.json`, `tokenizer.json`, `train` и `val`. Используется при создании нового эксперимента и подготовке отсутствующих данных для resume.
- `WORK_DIR`: рабочая папка Kaggle для конфигов, данных и результатов.

Имя эксперимента и имя конфига задаются отдельно в ветках 1 и 2. Начальные веса нужны только ветке 2, папка сохранённого эксперимента — только ветке 3.

In [1]:
from pathlib import Path

PROJECT_PATH = Path("/kaggle/input/datasets/wwip62/nlp-hw2-codegen").expanduser().resolve()
DATA_PATH = Path("/kaggle/input/datasets/wwip62/codegen-tokenized-bpe-16384/tokenized").expanduser().resolve()
WORK_DIR = Path("/kaggle/working").resolve()

TOKENIZER_PATH = DATA_PATH / "tokenizer.json"
DATA_DIR = WORK_DIR / "tokenized"

In [2]:
if not (PROJECT_PATH / "utilities" / "train_script.py").is_file():
    raise FileNotFoundError(f"Скрипт обучения не найден в репозитории: {PROJECT_PATH}")

print("Репозиторий:", PROJECT_PATH)
print("Исходные данные:", DATA_PATH)
print("Рабочая папка:", WORK_DIR)

Репозиторий: /kaggle/input/datasets/wwip62/nlp-hw2-codegen
Исходные данные: /kaggle/input/datasets/wwip62/codegen-tokenized-bpe-16384/tokenized
Рабочая папка: /kaggle/working


In [3]:
%pip install -q datasets tokenizers tqdm PyYAML "sacrebleu>=2.4,<3" wandb

Note: you may need to restart the kernel to use updated packages.


## Импорты

In [4]:
import os
import shutil
import subprocess
import sys

import torch
import yaml
from kaggle_secrets import UserSecretsClient

## Общие настройки

In [5]:
NUM_GPUS = 2

WANDB_PROJECT = "nlp-hw2"
WANDB_ENTITY = "62wip-hse-ml"

## Настройка W&B

In [6]:
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["WANDB_MODE"] = "online"
env["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## Подготовка данных и нового эксперимента

`prepare_data` копирует валидацию в рабочую папку для записи кеша и подключает train ссылкой. Функция вызывается при новом эксперименте и при отсутствии данных для resume.

`prepare_train_config` используется только в ветках 1 и 2: создаёт конфиг с абсолютными путями и именем эксперимента. Параметры warmup переопределяются независимо; `None` сохраняет соответствующее значение исходного конфига.

In [7]:
def prepare_data(data_dir=DATA_DIR):
    dataset_file = DATA_PATH / "dataset_dict.json"
    if not dataset_file.is_file():
        raise FileNotFoundError(f"Файл данных не найден: {dataset_file}")
    for split in ("train", "val"):
        if not (DATA_PATH / split).is_dir():
            raise FileNotFoundError(f"Выборка не найдена: {DATA_PATH / split}")

    data_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DATA_PATH / "dataset_dict.json", data_dir / "dataset_dict.json")
    shutil.copytree(DATA_PATH / "val", data_dir / "val", dirs_exist_ok=True)

    train_path = data_dir / "train"
    if train_path.is_symlink():
        train_path.unlink()
    shutil.copytree(DATA_PATH / "train", train_path, dirs_exist_ok=True)


def prepare_train_config(run_name, config_name):
    config_source = PROJECT_PATH / "experiment_configs" / config_name
    config_path = WORK_DIR / f"{run_name}_config.yaml"
    run_path = WORK_DIR / "experiment_runs" / run_name
    if run_path.exists():
        raise FileExistsError(f"Эксперимент уже существует: {run_path}. Нужно другое RUN_NAME или режим resume.")
    config = yaml.safe_load(config_source.read_text())
    if not TOKENIZER_PATH.is_file():
        raise FileNotFoundError(f"Токенайзер не найден: {TOKENIZER_PATH}")
    prepare_data()

    config["name"] = run_name
    config["data"]["tokenized_dir"] = str(DATA_DIR)
    config["data"]["tokenizer_path"] = str(TOKENIZER_PATH)
    if LR is not None:
        config["optimizer"]["params"]["lr"] = LR
    if MIN_LR is not None:
        config["scheduler"]["params"]["eta_min"] = MIN_LR
    if WARMUP_STEPS is not None or WARMUP_START_FACTOR is not None:
        warmup = config["scheduler"].get("warmup") or {}
        if WARMUP_STEPS is not None:
            warmup["steps"] = WARMUP_STEPS
        if WARMUP_START_FACTOR is not None:
            warmup["start_factor"] = WARMUP_START_FACTOR
        config["scheduler"]["warmup"] = warmup
    config_path.write_text(yaml.safe_dump(config, sort_keys=False))

    print("Конфиг:", config_path)
    print("Данные:", DATA_DIR)
    print("Токенайзер:", TOKENIZER_PATH)
    print("Результаты:", run_path)
    return config_path, run_path

## Запуск 3х экспериментов с лимитом времени

Обучаем с нуля три варианта Baseline++ с бюджетом по часу:
- **Контроль:** 5 encoder-слоёв, 7 decoder-слоёв, FFN=1024.
- **7/5:** переносим два слоя из декодера в энкодер.
- **FFN=1536:** сохраняем архитектуру 5/7 и увеличиваем ширину FFN.

Остальные настройки одинаковы. Сравниваем pass@1 на одной выборке из 512 вопросов. В час входят обучение, валидации и сохранения.

In [8]:
CONFIG_PATHS = [
    PROJECT_PATH / "experiment_configs/1_hour_exps/pilot_1h_control.yaml",
    PROJECT_PATH / "experiment_configs/1_hour_exps/pilot_1h_enc7_dec5.yaml",
    PROJECT_PATH / "experiment_configs/1_hour_exps/pilot_1h_ffn1536.yaml",
]
MAX_TRAIN_SECONDS = 3600

In [9]:
if not CONFIG_PATHS:
    raise ValueError("CONFIG_PATHS должен содержать хотя бы один путь к конфигу.")

experiments = []
run_names = set()
for source_path in CONFIG_PATHS:
    source_path = Path(source_path).expanduser().resolve()
    config = yaml.safe_load(source_path.read_text())
    run_name = config["name"]
    if not isinstance(run_name, str) or not run_name:
        raise ValueError(f"В конфиге не задано имя эксперимента: {source_path}")
    if run_name in run_names:
        raise ValueError(f"Имя эксперимента повторяется в очереди: {run_name}")
    run_names.add(run_name)

    run_path = WORK_DIR / "experiment_runs" / run_name
    if run_path.exists():
        raise FileExistsError(f"Эксперимент уже существует: {run_path}")
    config["data"]["tokenized_dir"] = str(DATA_DIR)
    config["data"]["tokenizer_path"] = str(TOKENIZER_PATH)
    config_path = WORK_DIR / f"{run_name}_config.yaml"
    experiments.append((config_path, config, run_path))

if not TOKENIZER_PATH.is_file():
    raise FileNotFoundError(f"Токенайзер не найден: {TOKENIZER_PATH}")
prepare_data()

for config_path, config, run_path in experiments:
    config_path.write_text(yaml.safe_dump(config, sort_keys=False))
    print(f"{config['name']}: {config_path} → {run_path}")

print(f"В очереди: {len(experiments)}; бюджет на эксперимент: {MAX_TRAIN_SECONDS} с")

pilot_1h_control: /kaggle/working/pilot_1h_control_config.yaml → /kaggle/working/experiment_runs/pilot_1h_control
pilot_1h_enc7_dec5: /kaggle/working/pilot_1h_enc7_dec5_config.yaml → /kaggle/working/experiment_runs/pilot_1h_enc7_dec5
pilot_1h_ffn1536: /kaggle/working/pilot_1h_ffn1536_config.yaml → /kaggle/working/experiment_runs/pilot_1h_ffn1536
В очереди: 3; бюджет на эксперимент: 3600 с


In [10]:
assert torch.cuda.device_count() >= NUM_GPUS, "Для запуска требуется GPU T4 ×2 в настройках Kaggle."

for number, (config_path, config, run_path) in enumerate(experiments, start=1):
    print(f"[{number}/{len(experiments)}] Запуск: {config['name']}", flush=True)
    command = [
        sys.executable, "-u", "-m", "torch.distributed.run",
        "--standalone", f"--nproc_per_node={NUM_GPUS}",
        "-m", "utilities.train_script",
        "--device", "cuda",
        "--config", str(config_path),
        "--max-train-seconds", str(MAX_TRAIN_SECONDS),
        "--wandb-project", WANDB_PROJECT,
    ]
    if WANDB_ENTITY:
        command += ["--wandb-entity", WANDB_ENTITY]

    result = subprocess.run(
        command,
        cwd=PROJECT_PATH,
        env={**env, "PROJECT_DIR": str(WORK_DIR)},
        check=False,
    )
    if result.returncode != 0:
        print(f"Очередь остановлена: {config['name']}, код выхода {result.returncode}.", flush=True)
        break
    print(f"Завершён: {config['name']} -> {run_path}", flush=True)
else:
    print("Все эксперименты в очереди завершены.")

[1/3] Запуск: pilot_1h_control



*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
Flattening the indices: 100%|██████████| 512/512 [00:00<00:00, 32330.91 examples/s]


Run: /kaggle/working/experiment_runs/pilot_1h_control; device: cuda:0; processes: 2; max_tokens per process: 8192


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 62wip (62wip-hse-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run mzis0hzv
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/experiment_runs/pilot_1h_control/wandb/run-20261023_155202-mzis0hzv
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run pilot_1h_control
wandb: ⭐️ View project at https://wandb.ai/62wip-hse-ml/nlp-hw2
wandb: 🚀 View run at https://wandb.ai/62wip-hse-ml/nlp-hw2/runs/mzis0hzv


Epoch 1, mini-epoch 1 | train loss = 3.2871, val loss = 2.2755
Examples: 100,020 | time = 209.0 s (train 163.2, val 45.8) | train/example = 0.001631 s
Small validation: token accuracy = 62.54%, top-5 = 78.12%
Heavy validation: pass@1 = 0.98% (5 / 512) | time = 44.5 s
Syntax valid = 77.15%, numeric return = 60.74%, no EOS = 9.18%
BLEU = 27.55, repetition = 27.85%, timeout = 0.00%
Epoch 1, mini-epoch 2 | train loss = 1.5119, val loss = 1.0686
Examples: 200,110 | time = 203.8 s (train 161.2, val 42.6) | train/example = 0.001611 s
Small validation: token accuracy = 77.85%, top-5 = 91.54%
Heavy validation: pass@1 = 11.91% (61 / 512) | time = 41.5 s
Syntax valid = 99.61%, numeric return = 88.28%, no EOS = 5.27%
BLEU = 36.97, repetition = 17.09%, timeout = 0.00%
Epoch 1, mini-epoch 3 | train loss = 0.9388, val loss = 0.7740
Examples: 300,009 | time = 207.4 s (train 163.4, val 44.0) | train/example = 0.001635 s
Small validation: token accuracy = 82.42%, top-5 = 94.31%
Heavy validation: pass@1 

...

Run: /kaggle/working/experiment_runs/pilot_1h_ffn1536; device: cuda:0; processes: 2; max_tokens per process: 8192


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 62wip (62wip-hse-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/experiment_runs/pilot_1h_ffn1536/wandb/run-20261023_175545-zz1wqz55
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run pilot_1h_ffn1536
wandb: ⭐️ View project at https://wandb.ai/62wip-hse-ml/nlp-hw2
wandb: 🚀 View run at https://wandb.ai/62wip-hse-ml/nlp-hw2/runs/zz1wqz55


Epoch 1, mini-epoch 1 | train loss = 3.2187, val loss = 2.2207
Examples: 100,020 | time = 224.3 s (train 177.1, val 47.2) | train/example = 0.001771 s
Small validation: token accuracy = 63.13%, top-5 = 78.89%
Heavy validation: pass@1 = 1.17% (6 / 512) | time = 46.0 s
Syntax valid = 85.35%, numeric return = 69.92%, no EOS = 4.49%
BLEU = 27.62, repetition = 24.58%, timeout = 0.00%
Epoch 1, mini-epoch 2 | train loss = 1.4751, val loss = 1.0377
Examples: 200,110 | time = 219.1 s (train 173.1, val 46.0) | train/example = 0.001729 s
Small validation: token accuracy = 78.38%, top-5 = 91.91%
Heavy validation: pass@1 = 12.89% (66 / 512) | time = 44.5 s
Syntax valid = 99.41%, numeric return = 89.65%, no EOS = 4.30%
BLEU = 35.82, repetition = 14.45%, timeout = 0.00%
Epoch 1, mini-epoch 3 | train loss = 0.9087, val loss = 0.7475
Examples: 300,009 | time = 225.9 s (train 177.9, val 48.0) | train/example = 0.001781 s
Small validation: token accuracy = 82.97%, top-5 = 94.58%
Heavy validation: pass@1 

wandb: updating run metadata
wandb: uploading history steps 16-16, summary, console lines 96-102
wandb: 
wandb: Run history:
wandb:              batch_index ▁▁▂▂▃▃▄▄▅▅▆▆▇▇▇██
wandb:             batches_seen ▁▁▂▂▃▃▄▄▅▅▆▆▇▇▇██
wandb:                     bleu ▁▃▅▆▇▇▇▇▇▇▇███▇▇█
wandb:                  correct ▁▃▄▄▅▆▆▆▇▇▇▇▇▇█▇█
wandb:                    epoch ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:            examples_seen ▁▁▂▂▃▃▄▄▅▅▆▆▇▇▇██
wandb: heavy_validation_seconds ▅▃▆▃▁▂▅▂▆▃▃▆▆█▂▂▅
wandb:                       lr ▁████████████████
wandb:               mini_epoch ▁▁▂▂▃▃▄▄▅▅▅▆▆▇▇██
wandb:      mini_epoch_examples ████████████████▁
wandb:                      +27 ...
wandb: 
wandb: Run summary:
wandb:              batch_index 12613
wandb:             batches_seen 12613
wandb:                     bleu 50.65863
wandb:           bleu_signature nrefs:var|case:mixed...
wandb:                  correct 275
wandb:                    epoch 1
wandb:            examples_seen 1609976
wandb: heavy_validation_seco

Завершён: pilot_1h_ffn1536 -> /kaggle/working/experiment_runs/pilot_1h_ffn1536
Все эксперименты в очереди завершены.


## Удаление рабочей копии данных

In [11]:
cleanup_dir = WORK_DIR / "tokenized"
if cleanup_dir.is_symlink():
    cleanup_dir.unlink()
elif cleanup_dir.exists():
    shutil.rmtree(cleanup_dir)
print("Рабочая копия данных удалена:", cleanup_dir)

Рабочая копия данных удалена: /kaggle/working/tokenized
